# Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import einsum, rearrange, reduce, repeat

# RoPE

## Sample q

In [86]:
# Sample q
q = torch.tensor(np.array([i*np.arange(11, 17) for i in range(1, 4)]))
q = q.unsqueeze(dim=0)
print(q)
print(q.shape)

B, T, d_head = q.shape

tensor([[[11, 12, 13, 14, 15, 16],
         [22, 24, 26, 28, 30, 32],
         [33, 36, 39, 42, 45, 48]]])
torch.Size([1, 3, 6])


## Theta

In [87]:
# theta_i = 1/(10000**( 2*(i-1)/d )) for all i in [1, 2, 3, ....d/2]
# exp term is [0, 2/d, 4/d, .......d-2/d]

head_dim = 6
theta_range = torch.arange(0, head_dim, 2)
theta_range

tensor([0, 2, 4])

In [88]:
theta = 1/10000**(theta_range/head_dim)
theta

tensor([1.0000, 0.0464, 0.0022])

## m_theta: Outer Product

In [ ]:
m = torch.arange(1, T+1)
m

tensor([1, 2, 3])

In [91]:
# Way1
m_theta = torch.outer(m, theta)
print(m_theta)
print(m_theta.shape)

tensor([[1.0000e+00, 4.6416e-02, 2.1544e-03],
        [2.0000e+00, 9.2832e-02, 4.3089e-03],
        [3.0000e+00, 1.3925e-01, 6.4633e-03]])
torch.Size([3, 3])


In [92]:
# Way2
m_theta_einsum = einsum(m, theta, "t, d -> t d")
print(m_theta_einsum)
print(m_theta_einsum.shape)

tensor([[1.0000e+00, 4.6416e-02, 2.1544e-03],
        [2.0000e+00, 9.2832e-02, 4.3089e-03],
        [3.0000e+00, 1.3925e-01, 6.4633e-03]])
torch.Size([3, 3])


# Repeat m_theta

In [77]:
# print(repeat(m_theta, "t d -> (t n) d", n=2))
# print('-'*50)
# print(repeat(m_theta, "t d -> (n t) d", n=2))

In [81]:
print(repeat(m_theta, "t d -> t (n d)", n=2))
print('-'*100)
print(repeat(m_theta, "t d -> t (d n)", n=2))

m_theta = repeat(m_theta, "t d -> t (d n)", n=2)
m_theta

tensor([[1.0000e+00, 4.6416e-02, 2.1544e-03, 1.0000e+00, 4.6416e-02, 2.1544e-03],
        [2.0000e+00, 9.2832e-02, 4.3089e-03, 2.0000e+00, 9.2832e-02, 4.3089e-03],
        [3.0000e+00, 1.3925e-01, 6.4633e-03, 3.0000e+00, 1.3925e-01, 6.4633e-03]])
----------------------------------------------------------------------------------------------------
tensor([[1.0000e+00, 1.0000e+00, 4.6416e-02, 4.6416e-02, 2.1544e-03, 2.1544e-03],
        [2.0000e+00, 2.0000e+00, 9.2832e-02, 9.2832e-02, 4.3089e-03, 4.3089e-03],
        [3.0000e+00, 3.0000e+00, 1.3925e-01, 1.3925e-01, 6.4633e-03, 6.4633e-03]])


tensor([[1.0000e+00, 1.0000e+00, 4.6416e-02, 4.6416e-02, 2.1544e-03, 2.1544e-03],
        [2.0000e+00, 2.0000e+00, 9.2832e-02, 9.2832e-02, 4.3089e-03, 4.3089e-03],
        [3.0000e+00, 3.0000e+00, 1.3925e-01, 1.3925e-01, 6.4633e-03, 6.4633e-03]])

## cos(m_theta) & sin(m_theta)

In [85]:
m_theta_cos = torch.cos(m_theta)
m_theta_sin = torch.sin(m_theta)
print(m_theta_cos)
print('-'*100)
print(m_theta_sin)

tensor([[ 0.5403,  0.5403,  0.9989,  0.9989,  1.0000,  1.0000],
        [-0.4161, -0.4161,  0.9957,  0.9957,  1.0000,  1.0000],
        [-0.9900, -0.9900,  0.9903,  0.9903,  1.0000,  1.0000]])
----------------------------------------------------------------------------------------------------
tensor([[0.8415, 0.8415, 0.0464, 0.0464, 0.0022, 0.0022],
        [0.9093, 0.9093, 0.0927, 0.0927, 0.0043, 0.0043],
        [0.1411, 0.1411, 0.1388, 0.1388, 0.0065, 0.0065]])


## Flip and negate first

In [104]:
# Copy
q_pair = q.detach().clone()
print(q_pair.shape)
print(q_pair)
print('-'*50)
# Reshape
q_pair = q_pair.view(B, T, -1, 2)
print(q_pair.shape)
print(q_pair)


torch.Size([1, 3, 6])
tensor([[[11, 12, 13, 14, 15, 16],
         [22, 24, 26, 28, 30, 32],
         [33, 36, 39, 42, 45, 48]]])
--------------------------------------------------
torch.Size([1, 3, 3, 2])
tensor([[[[11, 12],
          [13, 14],
          [15, 16]],

         [[22, 24],
          [26, 28],
          [30, 32]],

         [[33, 36],
          [39, 42],
          [45, 48]]]])


In [106]:
# Extract 1st and 2nd Col
t1 = q_pair[:, :, :, 0]
t2 = q_pair[:, :, :, 1]
print(t1.shape, t2.shape)
print('-'*50)
print(t1)
print('-'*50)
print(t2)

torch.Size([1, 3, 3]) torch.Size([1, 3, 3])
--------------------------------------------------
tensor([[[11, 13, 15],
         [22, 26, 30],
         [33, 39, 45]]])
--------------------------------------------------
tensor([[[12, 14, 16],
         [24, 28, 32],
         [36, 42, 48]]])


In [108]:
# Stack with negate first
q_filpped = torch.stack([(-1)*t2, t1], dim=-1)
print(q_filpped.shape)
print(q_filpped)

torch.Size([1, 3, 3, 2])
tensor([[[[-12,  11],
          [-14,  13],
          [-16,  15]],

         [[-24,  22],
          [-28,  26],
          [-32,  30]],

         [[-36,  33],
          [-42,  39],
          [-48,  45]]]])


In [109]:
# Final reshape
q_filpped = q_filpped.reshape(B, T, -1)
print(q_filpped.shape)
print(q_filpped)

torch.Size([1, 3, 6])
tensor([[[-12,  11, -14,  13, -16,  15],
         [-24,  22, -28,  26, -32,  30],
         [-36,  33, -42,  39, -48,  45]]])


## Final Rotated matrix for Q

In [112]:
q_cos = q * m_theta_cos
q_sin = q_filpped * m_theta_sin
q_rot = q_cos + q_sin
print(q_rot.shape)
print(q_rot)

torch.Size([1, 3, 6])
tensor([[[ -4.1543,  15.7398,  12.3364,  14.5881,  14.9655,  16.0323],
         [-30.9784,  10.0170,  23.2925,  30.2896,  29.8618,  32.1290],
         [-37.7501, -30.9828,  32.7930,  47.0066,  44.6888,  48.2898]]])


In [116]:
abc = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
abc

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

In [117]:
abc.shape

torch.Size([2, 2, 2])

In [ ]:
repeat(abc, "g t d -> (n g) t d", n=2)

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]],

        [[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

In [121]:
repeat(abc, "g t d -> (g n) t d", n=2)

tensor([[[1, 2],
         [3, 4]],

        [[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]],

        [[5, 6],
         [7, 8]]])

In [124]:
type(m_theta.device)

torch.device